In [ ]:
import numpy as np
import opendssdirect as dss
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import yadi.dss.model as dss_model
import yadi.dss.sensitivity as dss_sens
import yadi.dss.qsts as dss_qsts

In [ ]:
#Setup plot aesthetics
import matplotlib
%matplotlib inline
matplotlib.rc('text', usetex=True)
matplotlib.rc('text.latex', preamble=r'\usepackage{amsmath,amssymb}')
sns.set_theme(context='paper',style='ticks')

## Using OpenDSS in Python to analyze a 3-bus, unbalanced, OpenDSS test case.

Note that a 3-phase, 3-bus case implies 9 nodes in OpenDSS.

In [ ]:
#Load the case3_unbalanced file from PowerModelsDistribution.jl
case_name = 'case3_unbalanced'
cktfile = r"test_cases/{case_name}.dss".format(case_name=case_name)

### 1st Order Taylor Expansion Coefficients ("Voltage Sensitivity Matrices")

The object `dss_sens.OpenDSS_Sensitivities(cktfile)` allows you to form first-order Taylor expansion coefficients $\frac{\partial \mathbf{v}}{\partial \mathbf{p}}, \frac{\partial \mathbf{v}}{\partial \mathbf{q}}$ for any arbitrary opendss network model `ckfile`, such that
$$
\Delta \mathbf{v} \approx \frac{\partial \mathbf{v}}{\partial \mathbf{p}} \Delta \mathbf{p} + \frac{\partial \mathbf{v}}{\partial \mathbf{q}} \Delta \mathbf{q},
$$
where $\frac{\partial \mathbf{v}}{\partial \mathbf{p}}, \frac{\partial \mathbf{v}}{\partial \mathbf{q}} \in \mathbb{R}^{3n \times 3n}$, and $\Delta \mathbf{v}, \Delta \mathbf{p}, \Delta \mathbf{q} \in \mathbb{R}^{3 n}$

In [ ]:
sens = dss_sens.DSS_Sensitivities(cktfile,verbose=False)

#Get the taylor coefficients
dvdp = sens.get_svp()
dvdq = sens.get_svq()

#Plot the matrices of sensitivity coefficients including the slack nodes
fig,axes = plt.subplots(ncols=2)
h1 = sns.heatmap(dvdp["matrix"],ax=axes[0],center=0,cmap='mako',cbar=False)
h2 = sns.heatmap(dvdq["matrix"],ax=axes[1],center=0,cmap='mako')
axes[0].set_title(r"$\frac{\partial \mathbf{v}}{\partial \mathbf{p}}$")
axes[1].set_title(r"$\frac{\partial \mathbf{v}}{\partial \mathbf{q}}$")
plt.suptitle("Voltage sensitivities including slack nodes")

#Plot the matrices of sensitivity coefficients without the slack nodes
fig,axes = plt.subplots(ncols=2)
h1 = sns.heatmap(dvdp["matrix"][3:,3:],ax=axes[0],center=0,cmap='mako',cbar=False)
h2 = sns.heatmap(dvdq["matrix"][3:,3:],ax=axes[1],center=0,cmap='mako')
axes[0].set_title(r"$\frac{\partial \mathbf{v}}{\partial \mathbf{p}}$")
axes[1].set_title(r"$\frac{\partial \mathbf{v}}{\partial \mathbf{q}}$")
plt.suptitle("Voltage sensitivities without slack nodes")

### Nodal admittance matrix

In [ ]:
model = dss_model.DSS_Data(cktfile)
#Get the nodal admittance matrix and a vector of complex voltages in rectangular form.
Ynet,vnet = Y = model.get_node_ybus(True)

#Plot the admittance matrix
fig,axes = plt.subplots(ncols=2)
h1 = sns.heatmap(np.real(Ynet),ax=axes[0],center=0,cmap='mako',cbar=False)
h2 = sns.heatmap(np.imag(Ynet),ax=axes[1],center=0,cmap='mako')
axes[0].set_title(r"$\operatorname{Real}(\mathbf{Y})$")
axes[1].set_title(r"$\operatorname{Imag}(\mathbf{Y})$")
plt.suptitle(r"{case_name} Nodal Admittance Matrix".format(case_name=case_name))


#Plot the real and imaginary components of the bus voltages
plt.figure()
plt.stem(np.real(vnet),label=r"$\operatorname{Real}(\bar{v})$")
plt.stem(np.imag(vnet),label=r"$\operatorname{Imag}(\bar{v})$")
plt.legend()
plt.xlabel("Node index")
plt.ylabel("Nodal Voltage")
plt.title("Rectangular Nodal Voltages")

## Using Yadi to get timeseries data

- Parameters:
    - redirects (Circuit files/network model paths)
    - time_step (Time step of time series simulation (string))
    - simulation_steps (string) (Number of total simulation steps at time_step granularity)

In [ ]:
qsts = dss_qsts.DSS_Timeseries(
    redirects=cktfile,
    time_step="15m",
    simulation_steps=24*4
    )
qsts.run()

#Nodal current injections timeseries
I = qsts.currents_mvts

#Complex power injection timeseries
S = qsts.complex_powers_mvts

#Voltage magnitude timeseries
V = qsts.voltages_mvts


In [ ]:
V